# 11j — Weekly identifiability, and shrinkage of the dispersion random term

The per-week contact model (`constant_contacts=false`) fits a **hierarchical spatio-temporal GP**: a per-week level `c_t`, a matrix-normal structure field, and a per-age-pair-cell **dispersion random effect** (the *variance term*). Since 2026-08-02 that random effect is a **regularised horseshoe** (Piironen & Vehtari 2017, [arXiv:1707.01694](https://arxiv.org/pdf/1707.01694) eq. 11):

$$\log d_{ij,t} = \beta_{\ell(i,j),t} \;+\; \tau\cdot\tilde\lambda_{ij,t}\cdot z_{ij,t},\qquad \tilde\lambda^2 = \frac{c^2\lambda^2}{c^2+\tau^2\lambda^2},\qquad z\sim\mathcal N(0,1)$$

with a **window-global** shrinkage scale $\tau\sim\mathcal N^+(0,0.1^2)$ (it was one $\tau_t$ *per week* before), a **per-cell-per-week local scale** $\lambda_{ij,t}\sim\text{half-}t_3(0,1)$, and a **slab** $c^2\sim\text{InvGamma}(2,2)$ ($\nu=4$, $s=1$). The global scale shrinks every cell onto its block mean by default; a cell with enough data raises its own $\lambda$ to escape; the slab caps how far it can go, so $|\delta| \le c\,|z|$ exactly.

This notebook is **read-only** — it reconstructs everything from the cached `8j_s1_*` Stage-1 chains (no refit, no Stage 2) and produces two families of figure.

**§1 — Moment timelines.** The weekly **mean** degree `⟨k⟩` (`MeanNGM` C0) and **neighbourhood-mean** degree `⟨k²⟩/⟨k⟩·g` (`NeighbourhoodDegreeNGM` C0), with **90% CIs**, for a **contactee** of age **25-34** (bin 4) and **70+** (bin 7), one panel per **contactor** age *i*. The mean depends only on the level; the neighbourhood mean is driven by the second moment, hence by the dispersion random term. So **wider ribbons or larger week-to-week jumps in the neighbourhood line than the mean line** are the signature of that variance term — and show which weeks are (un)identifiable.

**§2 — Shrinkage.** Where the horseshoe actually acts, for both degree families. Note the two families' dispersion parameters read in **opposite directions**: for `unweighted-negbin` it is the NegBin dispersion **φ**, for `weighted-hweibull` it is the Weibull **shape κ**, where *smaller* κ means a heavier tail, i.e. *more* dispersion.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

default_plot_setting()

In [ ]:
# ── τ₀ TUNING STEP THIS NOTEBOOK VISUALISES ──────────────────────────────────────────────────
# Pinned explicitly rather than inherited from the FrameworkConfig default, so the figures always
# state which step they belong to even while that default moves during tuning. The chains for this
# step must already exist — fit them with `tune_tau0` (src/tune_tau0.jl) first.
TAU0 = 0.005

cfg  = FrameworkConfig(constant_contacts = false,   # per-week GP — required for weekly fluctuation
                       disp_re_scale_prior_unweighted = (0.0, TAU0),
                       disp_re_scale_prior_weighted   = (0.0, TAU0))
grid = cis_age_grid()
@assert grid.LAB[4] == "25-34" && grid.LAB[7] == "70+"
# token now encodes both τ₀ values, so it changes as you tune
println("token check   : ", contacts_label(cfg))

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)

ORIGIN = Date(2021, 5, 9)
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
# Fail loudly if this τ₀ step has not been fitted — otherwise every panel silently comes back blank.
for d in ("unweighted-negbin", "weighted-hweibull")
    f = joinpath("..", "dt_intermediate",
                 "8j_s1_$(d)_$(contacts_label(cfg))_$(ORIGIN)_h1.jld2")
    @assert isfile(f) "no chain for τ₀=$(TAU0), $(d) — run tune_tau0 for this step first ($(f))"
end

win = WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
wd  = load_window_data(win; grid = grid)

println("origin        : ", ORIGIN)
println("contacts token: ", contacts_label(cfg), "   (previous generation: ", CONTACTS_TOKEN_HD, ")")
println("contactees    : ", grid.LAB[4], " (j=4), ", grid.LAB[7], " (j=7)")
println("horseshoe     : tau0 negbin=", cfg.disp_re_scale_prior_unweighted[2],
        " hweibull=", cfg.disp_re_scale_prior_weighted[2],
        "  local df=", cfg.disp_rhs_local_df,
        "  slab df=", cfg.disp_rhs_slab_df, " scale=", cfg.disp_rhs_slab_scale)

In [ ]:
models = [NegBinAgePair(), HurdleWeibullAgePair()]
include("8j_viz_utils.jl")    # stage1_chain_path
include("10j_viz_utils.jl")   # reconstruct_rhs_components / _dispersion_draws, plot_rhs_globals
include("11j_viz_utils.jl")   # moment_timeline_stats, plot_moment_timeline, shrinkage panels
CONTACTEES = (4, 7)           # 25-34, 70+
println("degree models : ", degree_label.(models))

# Prior-predictive reference for the shrinkage statistics, PER FAMILY (τ₀ is per-family since
# 2026-08-02). `shrink` is a slab-vs-spike fraction, so its baseline is NOT zero — m_eff has a prior
# floor of order 1 per 49-cell week. Every m_eff panel below draws this band; quoting m_eff without
# it would overstate how many cells have escaped. NOTE this is the PRIOR only — it cannot be used to
# set τ₀ (see src/tune_tau0.jl, which measures the realised escape from fitted chains).
for dm in models
    PRI = prior_shrinkage_reference(cfg, dm)
    println(rpad(degree_label(dm), 18), " τ₀ = ", disp_tau0_prior(cfg, dm)[2],
            "   prior shrink median = ", round(PRI.shrink_q[1]; digits = 5),
            "   m_eff median = ", round(PRI.meff_q[1]; digits = 3),
            "  90% [", round(PRI.meff_q[2]; digits = 3), ", ", round(PRI.meff_q[3]; digits = 3), "]")
end

## §1 — Weekly mean vs neighbourhood-mean degree (90% CI)

Four figures = 2 contactees × {`unweighted-negbin`, `weighted-hweibull`}. Each is a 7-panel grid (contactor age *i*); steelblue = mean `⟨k⟩`, darkorange = neighbourhood `⟨k²⟩/⟨k⟩`, ribbons = 90% CI, dashed line = forecast origin. Faint gray bars (secondary right axis) = the per-cell sample size `n_pos = n_roster·(1−p⁰)` (positive contacts in that cell that week) — read the CI width against it: a wide ribbon over thin bars is a sample-starved, poorly-identified week. Saved to `../res/11j_moment_timeline_*`.

In [ ]:
# unweighted-negbin — contactee 25-34 then 70+
for j in CONTACTEES
    display(plot_moment_timeline(NegBinAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end

In [ ]:
# weighted-hweibull — contactee 25-34 then 70+ (K1 = (1−p⁰)·μW; neighbourhood carries the variance term)
for j in CONTACTEES
    display(plot_moment_timeline(HurdleWeibullAgePair(), j, ORIGIN, cfg, grid, raw, wd))
end

## §2 — How is the shrinkage happening?

The regularised horseshoe's per-cell **shrinkage factor** is

$$\text{shrink}_{ij,t} \;=\; \frac{c^2}{c^2+\tau^2\lambda_{ij,t}^2} \;=\; 1-\Big(\frac{\tau\tilde\lambda}{c}\Big)^{\!2}\ \in[0,1]$$

— **1** means the cell is fully shrunk onto its child/adult block mean, **0** means it has escaped into the slab and carries its own dispersion. The complementary *escape fraction* `1 − shrink` is what the panels below plot, and `m_eff = Σ_{ij}(1 − shrink)` is the effective number of unshrunk cells in a week.

> ⚠ This is the *model-internal* analogue of Piironen & Vehtari's $\kappa_j$, **not** the same number. The paper's $\kappa_j = 1/(1+n\sigma^{-2}\tau^2\tilde\lambda_j^2)$ measures shrinkage against the *data* information $n\sigma^{-2}$, which needs a Gaussian-likelihood approximation this model does not have; ours measures it against the *slab* scale $c^2$, which is exact. Both run 0→1 the same way — don't quote one as the other.

> ⚠ **`shrink` has a non-zero prior baseline** (≈0.998 at the prior medians), so `m_eff` has a prior floor of order 1 per 49-cell week, not 0. Every `m_eff` panel draws the grey prior-predictive band from `prior_shrinkage_reference`; read the posterior *against that band*, never in absolute terms.

Four diagnostics per degree model, then one old-vs-new comparison:

1. **Window-level scales** — τ (the spike width) and c (the slab ceiling) against their priors. τ near its prior is expected and harmless; **c** near its prior means nothing has escaped, because c is informed *only* through escaped cells.
2. **m_eff per week** vs the prior-predictive band and the 49-cell ceiling — the compact "how sparse is it" summary.
3. **Escape-fraction map** (7×7, origin week) beside the per-cell dispersion map. A uniformly dark grid is the *designed default*; the signal is a few individually bright cells.
4. **Ranked RE deviation** `δ = τλ̃z` across the 49 ordered cells — the spike-and-slab signature is a flat floor near 0 plus a handful of spikes toward the `±c` ceiling. A smooth staircase means it has degenerated into an ordinary hierarchical random effect.
5. **Escape vs sample size** — the headline check. Escape should rise with `n_pos`. **Escape concentrated at *low* `n_pos` is the warning sign**: prior noise leaking through exactly where there is no data, which then flows into `⟨k²⟩` and is amplified by the neighbourhood NGM.
6. **Old vs new** — within-block SD of log-dispersion under the previous flat `-hd` hierarchy versus the horseshoe. Both cache generations sit on disk under different tokens, so this needs no refit.

In [ ]:
# (1) window-level scales: tau (spike width) and c (slab ceiling) vs their priors
for dm in models
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_rhs_globals(lbl, ORIGIN, cfg; h = 1, weighted = is_weighted(dm))
    p === nothing || display(p)
end

In [ ]:
# (2) m_eff per week, against the prior-predictive band — is the horseshoe selecting anything?
for dm in models
    display(plot_meff_over_weeks(dm, ORIGIN, cfg, grid))
end

In [ ]:
# (3) escape-fraction map (origin week) beside the per-cell dispersion it produces.
#     unweighted-negbin -> phi ; weighted-hweibull -> Weibull shape kappa (smaller = MORE dispersed)
WK = cfg.smax + cfg.n_fit          # origin week = last window week (the one the NGM is frozen at)
for dm in models
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    display(plot_shrinkage_cells(dm, ORIGIN, cfg, grid; week_index = WK))
    display(plot_dispersion_cells(lbl, ORIGIN, cfg, grid;
                                  weighted = is_weighted(dm), week_index = WK))
end

In [ ]:
# (4) ranked RE deviation delta = tau*lamtilde*z over the 49 ordered cells (spike-and-slab signature)
for dm in models
    display(plot_re_ranked(dm, ORIGIN, cfg, grid; week_index = WK))
end

In [ ]:
# (5) HEADLINE CHECK — escape fraction vs per-cell informative sample size, all 49 x Tn cell-weeks.
#     Rising with n_pos = shrinkage is data-driven. Escape at LOW n_pos = prior noise leaking in.
for dm in models
    display(plot_shrinkage_vs_n(dm, ORIGIN, cfg, grid, raw))
end

In [ ]:
# (6) OLD vs NEW — within-block spread of log dispersion under the flat `-hd` hierarchy vs the
#     horseshoe. Reads both cache generations off disk (no refit). Lines are omitted with a warning
#     where a generation is missing: as of the 2026-08-02 pilot only this origin/h1 exists as `-rhs`.
for dm in models
    display(plot_within_block_sd(dm, ORIGIN, cfg, grid))
end

# The `-hd` chains also answer a question the scalar-tau choice depends on: did tau_t actually vary
# week to week? If it did, one window-level tau forces individual cells to absorb a week-level
# effect, and a per-week tau_t * lamtilde variant should be reconsidered.
for dm in models
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_tau_over_weeks(lbl, ORIGIN, cfg, win.all_weeks; h = 1, contacts = CONTACTS_TOKEN_HD)
    p === nothing || display(p)
end